# Pandas Essentials for Data Analysts

This notebook covers the main pandas operations every data analyst should know. It includes data loading, inspection, selection, transformation, grouping, missing value handling, date processing, string handling, merging, and export.

## 1. Setup and import

Start by importing pandas and setting a display option for better output readability.

In [ ]:
import pandas as pd
from io import StringIO

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print('pandas version:', pd.__version__)

## 2. Load data from common sources

Typical data sources are CSV, Excel, and JSON. Use pandas readers to load data into a DataFrame.

In [ ]:
csv_data = '''
order_id,customer,product,quantity,price,order_date,status
1001,Alice,Widget,4,20.5,2024-06-01,Delivered
1002,Bob,Gadget,2,15.0,2024-06-03,Returned
1003,Charlie,Widget,1,20.5,2024-06-05,Processing
1004,Dana,Doohickey,3,12.0,2024-06-07,Delivered
1005,Eli,Gadget,5,15.0,2024-06-10,Delivered
'''

df = pd.read_csv(StringIO(csv_data), parse_dates=['order_date'])
df.head()

## 3. Inspect the DataFrame

Use these methods to understand the dataset shape, types, and summary statistics.

In [ ]:
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())

df.info()

display(df.describe(include='all'))

## 4. Select columns and rows

Use `[]`, `.loc`, and `.iloc` to slice data. Boolean indexing is the primary way to filter rows.

In [ ]:
# Select a single column
product_series = df['product']
print(product_series.head(), '
')

# Select multiple columns
df[['customer', 'order_date', 'status']].head()

# Select rows by position
df.iloc[1:4]  # second through fourth rows

# Select rows by label and column names
df.loc[df['status'] == 'Delivered', ['order_id', 'customer', 'status']]

## 5. Add and transform columns

Create derived columns with vectorized arithmetic and mapping operations.

In [ ]:
df['total'] = df['quantity'] * df['price']
df['order_month'] = df['order_date'].dt.to_period('M')

status_map = {'Delivered': 'Complete', 'Returned': 'Failed', 'Processing': 'In Progress'}
df['status_label'] = df['status'].map(status_map)

df.head()

## 6. Grouping and aggregation

Group by one or more keys, then aggregate numeric values and counts.

In [ ]:
# Total revenue by product
revenue_by_product = df.groupby('product', as_index=False).agg(
    total_revenue=('total', 'sum'),
    average_quantity=('quantity', 'mean'),
    orders=('order_id', 'count')
)
revenue_by_product

# Revenue by customer and order month
monthly_customer = df.groupby(['customer', 'order_month'], as_index=False)['total'].sum()
monthly_customer

## 7. Sort and rank data

Sort rows to identify top values and use rank when needed.

In [ ]:
# Sort by total revenue descending
df.sort_values(by='total', ascending=False).head()

# Rank orders by total in the original order
df['total_rank'] = df['total'].rank(method='dense', ascending=False).astype(int)
df[['order_id', 'customer', 'total', 'total_rank']]

## 8. Handle missing values

Missing data is common. Inspect and fill or drop values depending on the use case.

In [ ]:
df_missing = df.copy()
df_missing.loc[2, 'price'] = None
df_missing.loc[4, 'customer'] = None

print('Missing counts:')
print(df_missing.isna().sum())

# Fill numeric missing values with mean and forward-fill strings
df_missing['price'] = df_missing['price'].fillna(df_missing['price'].mean())
df_missing['customer'] = df_missing['customer'].fillna(method='ffill')

df_missing

## 9. Work with dates

Date columns can be parsed and used to create features such as day of week, month, and business logic filters.

In [ ]:
df['order_day'] = df['order_date'].dt.day_name()
df['order_week'] = df['order_date'].dt.isocalendar().week

df[['order_id', 'order_date', 'order_day', 'order_week']].head()

# Filter orders placed in June 2024
june_orders = df[df['order_date'].dt.month == 6]
june_orders

## 10. String operations

Use vectorized string methods for cleaning and extracting text data.

In [ ]:
df['product_clean'] = df['product'].str.lower()
df['customer_initial'] = df['customer'].str[0]

df[['product', 'product_clean', 'customer', 'customer_initial']]

## 11. Combine and join data

Merging and concatenating are essential when bringing multiple tables together.

In [ ]:
customer_data = pd.DataFrame({
    'customer': ['Alice', 'Bob', 'Charlie', 'Dana', 'Eli'],
    'region': ['North', 'South', 'East', 'West', 'South']
})

merged = df.merge(customer_data, on='customer', how='left')
merged[['order_id', 'customer', 'region', 'total']].head()

# Concatenate new rows to the original DataFrame
new_orders = pd.DataFrame([
    {'order_id': 1006, 'customer': 'Fiona', 'product': 'Widget', 'quantity': 2, 'price': 20.5, 'order_date': pd.Timestamp('2024-06-12'), 'status': 'Delivered', 'total': 41.0, 'order_month': pd.Period('2024-06'), 'status_label': 'Complete'}
])
all_orders = pd.concat([df, new_orders], ignore_index=True)
all_orders.tail()

## 12. Export results

Save a cleaned or aggregated DataFrame to CSV or Excel for reporting and sharing.

In [ ]:
output_csv = 'pandas_essentials_output.csv'
aggregated = df.groupby('status_label', as_index=False)['total'].sum()
aggregated.to_csv(output_csv, index=False)
print('Saved:', output_csv)
aggregated